In [ ]:
import os, warnings, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, balanced_accuracy_score
warnings.filterwarnings('ignore')

D = '/content/'
tr, te = pd.read_csv(D+'train.csv'), pd.read_csv(D+'test.csv')
lab = lambda s: s.astype(str).str.lower().isin(['true', '1', '1.0'])
ID = ['flow_key','timeframe_id','ip_a','ip_b','port_a','port_b']
FEAT = [c for c in te.columns if c != 'is_attack' and c not in ID
        and not c.startswith(('target_','source_','flow_share_'))]
X = lambda df: df[FEAT].apply(pd.to_numeric, errors='coerce')
Xte, yte = X(te), lab(te['is_attack'])
ben = tr[~lab(tr['is_attack'])]
print('features', len(FEAT), '| benign train', len(ben), '| test attack/benign', yte.sum(), (~yte).sum())

MODELS = {
 'RF':  lambda: RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
 'LR':  lambda: LogisticRegression(max_iter=2000, C=1, class_weight='balanced', random_state=42),
 'SVM': lambda: LinearSVC(C=1, class_weight='balanced', max_iter=5000, random_state=42)}

def run(name, att):
    trn = pd.concat([att.assign(is_attack=True), ben.assign(is_attack=False)])
    out = []
    for m, mk in MODELS.items():
        p = make_pipeline(SimpleImputer(strategy='constant', fill_value=0), StandardScaler(), mk())
        pred = p.fit(X(trn), lab(trn['is_attack'])).predict(Xte)
        tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[False, True]).ravel()
        out.append(dict(Set=name, Clf=m, TN=tn, FP=fp, FN=fn, TP=tp,
                        FPR=round(fp/(tn+fp), 3), F1=round(f1_score(yte, pred), 3),
                        BA=round(balanced_accuracy_score(yte, pred), 3)))
    return out

rows = run('TRTR', tr[lab(tr['is_attack'])])
print('TRTR anchor (expect (192,2),(192,2),(191,3)):', [(int(r['TN']), int(r['FP'])) for r in rows])

rd = lambda f: pd.read_csv(D+f)
for g in ['Llama', 'Falcon', 'Qwen']:
    rows += run(f'{g} (A)', pd.concat([rd(f'{g.lower()}_run1.csv'), rd(f'{g.lower()}_run2.csv')]))
if os.path.exists(D+'Fused_A_combined.csv'):
    rows += run('Fused (A)', rd('Fused_A_combined.csv'))

res = pd.DataFrame(rows)
print(res.to_string(index=False))
a = res[res.Set.str.contains(r'\(A\)')]
print(f"\nSetting A: max FP = {a.FP.max()} of 194 (FPR {a.FP.max()/194:.1%}), max FN = {a.FN.max()} of 1346")
res.to_csv(D+'confusion_setting_A.csv', index=False)

print('\n% LaTeX rows: Set & Clf & TN & FP & FN & TP & FPR')
for _, r in res.iterrows():
    print(f"{r.Set} & {r.Clf} & {r.TN} & {r.FP} & {r.FN} & {r.TP} & {r.FPR:.3f} \\\\")

In [ ]:
import matplotlib.pyplot as plt

# Fused A combined counts from tstr_hard_fused_BA.csv (same Oct 23 pipeline)
fused = pd.DataFrame([
    dict(Set='Fused (A)', Clf='RF',  TN=194, FP=0, FN=223, TP=1123),
    dict(Set='Fused (A)', Clf='LR',  TN=193, FP=1, FN=263, TP=1083),
    dict(Set='Fused (A)', Clf='SVM', TN=194, FP=0, FN=306, TP=1040)])
fused['FPR'] = (fused.FP / 194).round(3)
show = pd.concat([res[['Set','Clf','TN','FP','FN','TP','FPR']], fused], ignore_index=True)

# 1) Summary table
summary = show.pivot(index='Set', columns='Clf', values=['FP','FN','FPR'])
summary = summary.reindex(['TRTR','Llama (A)','Falcon (A)','Qwen (A)','Fused (A)'])
summary = summary.reindex(columns=['RF','LR','SVM'], level=1)
print('False positives (of 194 benign), false negatives (of 1,346 attacks), FPR\n')
display(summary)
print(f"Setting A max FP: {show[show.Set!='TRTR'].FP.max()} of 194 "
      f"({show[show.Set!='TRTR'].FP.max()/194:.1%})")

# 2) Confusion-matrix grid
sets = ['TRTR','Llama (A)','Falcon (A)','Qwen (A)','Fused (A)']
fig, axes = plt.subplots(len(sets), 3, figsize=(9, 2.6*len(sets)))
for i, s in enumerate(sets):
    for j, c in enumerate(['RF','LR','SVM']):
        r = show[(show.Set==s) & (show.Clf==c)].iloc[0]
        m = np.array([[r.TN, r.FP], [r.FN, r.TP]])
        ax = axes[i, j]
        ax.imshow(m, cmap='Blues')
        for (y, x), v in np.ndenumerate(m):
            ax.text(x, y, int(v), ha='center', va='center',
                    color='white' if v > m.max()/2 else 'black', fontsize=11)
        ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred benign','Pred attack'], fontsize=8)
        ax.set_yticks([0, 1]); ax.set_yticklabels(['True benign','True attack'], fontsize=8)
        ax.set_title(f'{s} – {c}  (FPR {r.FP/194:.1%})', fontsize=9)
plt.tight_layout()
plt.savefig('/content/confusion_setting_A.png', dpi=200, bbox_inches='tight')
plt.show()